In [ ]:
"""
Imports

"""

In [ ]:
import json
import math
from itertools import groupby
from typing import Callable, Dict, List, Optional, Set, Tuple, Type, Union

import numpy as np
import PIL
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from safetensors.torch import safe_open
    from safetensors.torch import save_file as safe_save

    safetensors_available = True
except ImportError:
    from .safe_open import safe_open
import argparse
import hashlib
import inspect
import itertools
import math
import os
import random
import re
from pathlib import Path
from typing import Optional, List, Literal

import torch
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.checkpoint
from diffusers import (
    AutoencoderKL,
    DDPMScheduler,
    StableDiffusionPipeline,
    UNet2DConditionModel,
)
from diffusers.optimization import get_scheduler
from huggingface_hub import HfFolder, Repository, whoami
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import CLIPTextModel, CLIPTokenizer
import wandb
import random
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

from PIL import Image
from torch import zeros_like
from torch.utils.data import Dataset
from torchvision import transforms
import glob
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
from PIL import ImageOps

In [ ]:
"""
We define the modules to be replaced in the UNET and the text encoder (the base code allows for fine-tuning the text encoder, so
I left this option in, but for pivotal tuning we don't need it), as well as a flag that will be useful later for identifying the text encoder embeddings
in the metadata.

"""

UNET_TARGET_REPLACE = {"UNet2DConditionModel"}

TEXT_ENCODER_TARGET_REPLACE = {"CLIPTextModel"}

EMBED_FLAG = "<embed>"

In [ ]:
"""
Loading models, creating the dataloader and various token lists

"""

In [ ]:
"""
Class used to resize images to squares by adding black bands on the shorter sides

"""
class PadToSquare:
    def __init__(self, fill=0):
        self.fill = fill

    def __call__(self, img):
        w, h = img.size
        max_dim = max(w, h)
        padding = (
            (max_dim - w) // 2,  # gauche
            (max_dim - h) // 2,  # haut
            (max_dim - w + 1) // 2,  # droite
            (max_dim - h + 1) // 2   # bas
        )
        return ImageOps.expand(img, padding, fill=self.fill)

In [ ]:
class InversionDataset(Dataset):

    """
    The class inherits from pytorch's Dataset, which expects a len method and a getitem method.

    init:
    Builds a list from the images given in instance_data_root: [“./images/cat1.jpg”, “./images/cat2.png”, “./images/cat3.jpeg”]
    Constructs a list of titles from the image titles: [“cat1”,“cat2”,cat3]
    Resizes the images to size, crops the images so that they are square, normalizes the pixel values between [-1,1]
    and transforms the image into a tensor


    get_item:
    Returns a dictionary example or example[“instance_images”] = the image transformed into a tensor and  example[“instance_prompt_ids”] = the list
    of IDs of the prompt tokens, which is the name given to the image in the folder

    """

    def __init__(
        self,
        instance_data_root,
        tokenizer,
        size=512,
        resize=True,
        token_map: Optional[dict] = None,
        do_regularization: Optional[bool] = False,
        regularization_dataset_length : Optional[int] = None,

    ):
        if do_regularization :
          print("Creating the dataset for regularization")
        else :
          print("Creating the dataset for inversion/tuning")

        self.size = size
        self.tokenizer = tokenizer
        self.resize = resize

        instance_data_root = Path(instance_data_root)
        if not instance_data_root.exists():
            raise ValueError("Instance images root doesn't exists.")

        self.instance_images_path = []

        #Le résultat est une liste de chemins d'images ["./images/cat1.jpg", "./images/cat2.png", "./images/cat3.jpeg"
        #glob.glob() est utilisé pour construire automatiquement une liste de fichiers
        #correspondant à un motif de nom de fichier, par exemple toutes les images dans un dossier avec certaines extensions
        self.instance_images_path = (
            glob.glob(str(instance_data_root) + "/*.jpg")
            + glob.glob(str(instance_data_root) + "/*.png")
            + glob.glob(str(instance_data_root) + "/*.jpeg")
        )

        print("Début de la liste d'images %s" % self.instance_images_path[:3])

        if do_regularization :
          #On génère une liste de captions à partir des noms de fichiers
          # Pour `./images/cat1.jpg`, on obtient la caption `"cat1"`.
          self.captions = [
             "an anime illustration of character woman with long blue hair" for x in self.instance_images_path
          ]
          print("Début des captions %s" % self.captions[:3])

        else :
          self.captions = [
              "an anime illustration of ichigo" for x in self.instance_images_path
          ]
          # self.captions = [
          #     x.split("/")[-1].split(".")[0] for x in self.instance_images_path
          # ]
          print("Début des captions %s" % self.captions[:3])

        self.instance_images_path = sorted(self.instance_images_path)

        #on multiplie le nombre d'images/captions pour que le dataset des images soit de même taille que le dataset pour la locality regularization
        #ce qui facilite le parcours des dataloaders que l'on verra après
        if regularization_dataset_length is not None :
          print ("Taille originale des captions %s"% (len(self.captions)))
          self.captions *= int(regularization_dataset_length/len(self.captions))
          print ("Nouvelle taille des captions %s"% (len(self.captions)))
          print ("Taille originale des images %s"% (len(self.instance_images_path)))
          self.instance_images_path *= int(regularization_dataset_length/len(self.instance_images_path))
          print ("Nouvelle originale des images %s"% (len(self.instance_images_path)))

        self._length = len(self.instance_images_path)

        #suite de transformations appliquées à une image
        self.image_transforms = transforms.Compose(
            [   #redimensionne l'image à la taille size si resize est vrai
                #permet de choisir à quel point on veut analyser des images de haute qualité, sachant que plus la qualité est grande plus ca prend de place
                #en mémoire
                PadToSquare(fill=0),
                transforms.Resize(
                    size, interpolation=transforms.InterpolationMode.BILINEAR
                )
                if resize
                else transforms.Lambda(lambda x: x),
                #crop l'image pour qu'elle soit carrée
                #avoir toutes les images de la même forme est nécessaire pour que le dataloader puisse créer un tensor des images
                transforms.CenterCrop(size),
                #convertit l'image en tensor
                transforms.ToTensor(),
                #Normalise les pixels : `x = (x - 0.5) / 0.5`
                #Résultat : valeurs entre `[-1, 1]` au lieu de `[0, 1]`
                #Cette normalisation est **standard dans les modèles de diffusion** (ex : Stable Diffusion) qui attendent des inputs entre -1 et 1.
                transforms.Normalize([0.5], [0.5]),
            ]
        )
        self.token_map = token_map

    def __len__(self):
        return self._length

    def __getitem__(self, index):

        example = {}

        #on place l'image index dans le dictionnaire example

        instance_image = Image.open(
            self.instance_images_path[index]
        )
        if not instance_image.mode == "RGB":
            instance_image = instance_image.convert("RGB")

        example["instance_images"] = self.image_transforms(instance_image)

        #on place la liste d'ids du prompt dans le dictionnaire example

        text = self.captions[index]

        #Si on a une token_map : {"<tok1>" : "nom_du_personnage"}, on remplace dans le texte "nom_du_personnage" par "<tok1>"
        #on fait car on ne peut pas utiliser de caractères spéciaux pour nommer une image
        if self.token_map is not None:
            for token, value in self.token_map.items():
                text = text.replace(token, value)

        #tokenize le texte "image1" → ["image", "1"], convertit les tokens en IDs (par ex. [2034, 129]) et retourne ces IDs
        example["instance_prompt_ids"] = self.tokenizer(
            text,
            padding="do_not_pad",
            truncation=True,
            max_length=self.tokenizer.model_max_length,
        ).input_ids

        return example

In [ ]:
def get_models(
    pretrained_model_name_or_path,
    safeloras_path,
    revision,
    placeholder_tokens: List[str],
    initializer_tokens: List[str],
    device="cuda:0",
    do_pivotal_tuning = False
):
    """
    Parameters:
      placeholder_tokens = [“<tok1>”,“<tok2>”] : the list of new tokens we want to teach the text encoder
      initializer_tokens = [“<zeros>”,“<rand-0.5>”] : flags corresponding to different initialization methods for the embeddings of these new tokens

    Actions:
      Adds placeholder_tokens to the tokenizer
      Adds placeholder_token ids to placeholder_token_ids
      Increases the size of text_encoder to match the new size of the tokenizer
      Initializes the embeddings of the new tokens according to initializer_tokens
      Returns the models loaded on cuda and the placeholder_token_ids list

    """

    tokenizer = CLIPTokenizer.from_pretrained(
        pretrained_model_name_or_path,
        subfolder="tokenizer",
        revision=revision,
    )

    text_encoder = CLIPTextModel.from_pretrained(
        pretrained_model_name_or_path,
        subfolder="text_encoder",
        revision=revision,
    )

    vae = AutoencoderKL.from_pretrained(
        pretrained_model_name_or_path,
        subfolder="vae",
        revision=revision,
    )
    unet = UNet2DConditionModel.from_pretrained(
        pretrained_model_name_or_path,
        subfolder="unet",
        revision=revision,
    )

    placeholder_token_ids = []
    initializer_token_ids = []

    if do_pivotal_tuning :
      placeholder_tokens = []
      initializer_tokens = []
      #Si on fait le pivotal tuning, on ajoute les embeddings appris avec l'inversion
      safeloras = safe_open(safeloras_path, framework="pt", device="cpu")
      tok_dict = parse_safeloras_embeds(safeloras)
      apply_learned_embed_in_clip(
          tok_dict,
          text_encoder,
          tokenizer,
      )

    else :

      #construit la liste des placeholder_tokens et des initializer_tokens à partir des chaînes de caractères passées en argument
      if len(placeholder_tokens) == 0:
          placeholder_tokens = []
          print("PTI : Placeholder Tokens not given, using null token")
      else:
          placeholder_tokens = placeholder_tokens.split("|")

          assert (
              sorted(placeholder_tokens) == placeholder_tokens
          ), f"Placeholder tokens should be sorted. Use something like {'|'.join(sorted(placeholder_tokens))}'"

      if initializer_tokens is None:
          print("PTI : Initializer Tokens not given, doing random inits")
          initializer_tokens = ["<rand-0.017>"] * len(placeholder_tokens)
      else:
          initializer_tokens = initializer_tokens.split("|")

      assert len(initializer_tokens) == len(
          placeholder_tokens
      ), "Unequal Initializer token for Placeholder tokens."


      print("PTI : Placeholder Tokens", placeholder_tokens)
      print("PTI : Initializer Tokens", initializer_tokens)
      for token, init_tok in zip(placeholder_tokens, initializer_tokens):
          num_added_tokens = tokenizer.add_tokens(token)
          if num_added_tokens == 0:
              raise ValueError(
                  f"The tokenizer already contains the token {token}. Please pass a different"
                  " `placeholder_token` that is not already in the tokenizer."
              )

          placeholder_token_id = tokenizer.convert_tokens_to_ids(token)

          placeholder_token_ids.append(placeholder_token_id)

          #On redimensionne le text_encoder car la dimension du vocabulaire s'est agrandie
          text_encoder.resize_token_embeddings(len(tokenizer))
          #La matrice de poids du text_encoder
          token_embeds = text_encoder.get_input_embeddings().weight.data
          #initialisation au hasard
          if init_tok.startswith("<rand"):
              # <rand-"sigma">, e.g. <rand-0.5>
              sigma_val = float(re.findall(r"<rand-(.*)>", init_tok)[0])

              token_embeds[placeholder_token_id] = (
                  torch.randn_like(token_embeds[0]) * sigma_val
              )
              print(
                  f"Initialized {token} with random noise (sigma={sigma_val}), empirically {token_embeds[placeholder_token_id].mean().item():.3f} +- {token_embeds[placeholder_token_id].std().item():.3f}"
              )
              print(f"Norm : {token_embeds[placeholder_token_id].norm():.4f}")
          #initialisation avec un embedding nul
          elif init_tok == "<zero>":
              token_embeds[placeholder_token_id] = torch.zeros_like(token_embeds[0])
          #initialisation avec l'embedding d'un mot déjà appris
          else:
              print("Remplacement de l'embedding du token %s par l'embedding de %s " % (token,init_tok))
              token_ids = tokenizer.encode(init_tok, add_special_tokens=False)
              # Check if initializer_token is a single token or a sequence of tokens
              if len(token_ids) > 1:
                  raise ValueError("The initializer token must be a single token.")

              initializer_token_id = token_ids[0]
              initializer_token_ids.append(initializer_token_id)
              token_embeds[placeholder_token_id] = token_embeds[initializer_token_id]

    #les modèles sont par défaut chargés sur le cpu donc il faut explicitement les mettre sur cuda
    return (
        text_encoder.to(device),
        vae.to(device),
        unet.to(device),
        tokenizer,
        placeholder_tokens,
        initializer_tokens,
        placeholder_token_ids,
        initializer_token_ids
    )

In [ ]:
def text2img_dataloader(
    train_dataset,
    train_batch_size,
    tokenizer,
    vae,
    text_encoder,
):
    #explication de latents = latents * 0.18215
    """
    We iterate through the example dictionaries in InversionDataset with a for idx loop, which is possible thanks to the getitem function.
    We create a list cached_latents_dataset = [] that contains the example objects where the image has been replaced by a latent with the vae.
    We use the Dataloader function, which:
        creates sublists of cached_latents_dataset of size batch_size
        applies collate_fn to them, which constructs a tensor of images and a tensor of captions and returns a batch dictionary of the two
        returns a train_dataloader iterator that can be traversed to obtain the batches, and at the end of a complete traversal of the iterator, the batches
        are recreated in a different order


    """


    #permet de ne pas construire de graphe à partir de la transformation des images en latent par le vae
    #un graphe des gradients est construit automatiquement si on fait des opérations sur un tensor tant qu'on ne précise pas torch.no_grad()
    with torch.no_grad() :
      cached_latents_dataset = []
      #tdqm rajoute la barre de progression
      for idx in tqdm(range(len(train_dataset))):
          batch = train_dataset[idx]
          #batch["instance_images"] : un Tensor image de taille (3, H, W) (format image).
          #unsqueeze(0) : Les modèles comme vae.encode() attendent des batches d’images, même si c’est un seul élément. Donc on ajoute une dimension "batch = 1".
          #to(vae.device) : le vae va faire des opérations sur le tenseur qui doit donc être sur le même device
          #latent_dist.sample() : le vae encode l'image en distribution latente. On échantillonne un latent vector depuis cette distribution.
          latents = vae.encode(batch["instance_images"].unsqueeze(0).to(vae.device)
          ).latent_dist.sample()
          latents = latents * 0.18215
          #on remplace les images par les latents
          batch["instance_images"] = latents.squeeze(0)
          cached_latents_dataset.append(batch)
      idx = random.randint(0,len(train_dataset)-1)
      batch = train_dataset[idx]
      print("Une image en cache")
      img_tensor = batch["instance_images"]
      img = TF.to_pil_image(img_tensor)
      plt.imshow(img)
      plt.axis("off")
      plt.show()
      print("Caption de l'image %s" % tokenizer.convert_ids_to_tokens(batch["instance_prompt_ids"]))

    #examples est une sous liste de cached_latents_dataset de taille batch_size
    def collate_fn(examples):
        #création d'un tensor pour les images : (batch_size, C, H, W)
        pixel_values = [example["instance_images"] for example in examples]
        pixel_values = torch.stack(pixel_values)

        #création d'un tensor pour les ids : (batch_size, sequence_length)
        #pour cela on fait un padding de la taille du tokenizer qui a la même utilité que le crop pour les images
        #ex :  "A dog" ->	[1547, 21] ->	[1547, 21, 0, ..., 0]
        input_ids = [example["instance_prompt_ids"] for example in examples]
        input_ids = tokenizer.pad(
            {"input_ids": input_ids},
            padding="max_length",
            max_length=tokenizer.model_max_length,
            return_tensors="pt",
        ).input_ids

        batch = {
            "input_ids": input_ids,
            "pixel_values": pixel_values,
        }

        return batch



    train_dataloader = torch.utils.data.DataLoader(
        cached_latents_dataset,
        batch_size=train_batch_size,
        # le DataLoader lit les données dans un ordre différent à chaque epoch, ce qui est une forme de randomisation très utile pour éviter l’overfitting et améliorer la généralisation.
        #PyTorch crée un RandomSampler : indices aléatoires [2, 0, 3, 1]
        shuffle=True,
        #Puis crée des mini-batches en regroupant les indices : [[2, 0], [3, 1]] si batch_size=2
        collate_fn=collate_fn,
    )

    return train_dataloader

In [ ]:
def prepare_dataset(
    instance_data_dir: str,
    regularization_data_dir : str,
    pretrained_model_name_or_path: str,
    output_dir: str,
    revision: Optional[str] = None,
    placeholder_tokens: str = "",
    initializer_tokens: Optional[str] = None,
    seed: int = 42,
    resolution: int = 512,
    train_batch_size: int = 2,
    device="cuda:0",
    token_map: Optional[dict] = None,
    safeloras_path: Optional[str] = None,
    do_pivotal_tuning : Optional[bool] = False,
    do_regularization : Optional[bool] = False,
):

    """
    Actions:
      creation of the list of placeholder_tokens and initializer_tokens from the character strings passed as arguments:
          e.g. placeholder_tokens: “<s1>|<s2>”
          e.g. initializer_tokens: “<rand-0.5>|<zeros>”
      creation of the placeholder_token_ids list and loading of models with get_models
      creation of the scheduler with DDPMScheduler.from_config
      creation of the dataset with InversionDataset
      creation of the dataloader with text2img_dataloader
      create an index_no_updates list of the same size as tokenizer where all IDs except those in placeholder_token_ids are True
      which indicates the embeddings that should not be trained

    Returns:
      the models, the dataloader, the various token lists, index_no_updates

    """


    torch.manual_seed(seed)

    if output_dir is not None:
        os.makedirs(output_dir, exist_ok=True)

    #Récupération des modèles

    text_encoder, vae, unet, tokenizer, placeholder_tokens, initializer_tokens, placeholder_token_ids, initializer_token_ids = get_models(
          pretrained_model_name_or_path,
          safeloras_path,
          revision,
          placeholder_tokens,
          initializer_tokens,
          device=device,
          do_pivotal_tuning=do_pivotal_tuning
      )


    noise_scheduler = DDPMScheduler.from_config(
        pretrained_model_name_or_path, subfolder="scheduler"
    )


    #Création du tuning_dataloader
    inversion_dataset = InversionDataset(
        instance_data_root=instance_data_dir,
        tokenizer=tokenizer,
        size=resolution,
        token_map=token_map,
        do_regularization = False,
        regularization_dataset_length = 96,
    )

    inversion_dataloader = text2img_dataloader(
        inversion_dataset,
        train_batch_size,
        tokenizer,
        vae,
        text_encoder,
    )

    if do_regularization :
      #Création du regularization_dataloader
      regularization_dataset = InversionDataset(
          instance_data_root=regularization_data_dir,
          tokenizer=tokenizer,
          size=resolution,
          token_map=token_map,
          do_regularization = True,
      )

      regularization_dataloader = text2img_dataloader(
          regularization_dataset,
          train_batch_size,
          tokenizer,
          vae,
          text_encoder,
      )

    #on peut supprimer le vae qui ne sera plus utilisé
    del vae
    torch.cuda.empty_cache()

    #liste de même taille que tokenizer où tous les éléments valent True
    index_no_updates = torch.arange(len(tokenizer)) != -1

    #on met False sur les éléments à update
    for tok_id in placeholder_token_ids:
        index_no_updates[tok_id] = False

    if do_regularization :
      return placeholder_tokens, initializer_tokens, placeholder_token_ids, initializer_token_ids, text_encoder, unet, tokenizer, noise_scheduler,  index_no_updates, inversion_dataloader, regularization_dataloader
    else :
      return placeholder_tokens, initializer_tokens, placeholder_token_ids, initializer_token_ids, text_encoder, unet, tokenizer, noise_scheduler,  index_no_updates, inversion_dataloader


In [ ]:
"""
Functions for training in text inversion or pivotal tuning

"""

In [ ]:
def loss_step(
    inversion_batch,
    unet,
    text_encoder,
    scheduler,
    t_mutliplier=1.0,
    global_step = 0,
    control_step = 100,
    do_regularization : Optional[bool] = False,
    regularization_batch : Optional[torch.utils.data.DataLoader] = None,
):
    #utilité de t_mutliplier
    """
    Calculate the error between the noise added to the batch latents and the error predicted by the UNET

    Actions:
      Create a noise tensor composed of Gaussian noise for each latent in the batch
      Create a timesteps tensor composed of time steps randomly drawn between 1 and scheduler.config.num_train_timesteps for each latent in the batch
      Create a noisy_latents tensor composed of the latents to which Gaussian noise has been added according to the time step, using the diffusion model formula.
      Create an encoder_hidden_states tensor composed of the embeddings corresponding to the prompt's inputs_ids.
      Apply the UNET to the noisy_latents, timesteps, and encoder_hidden_states tensors.
      Calculation of the loss between the sample result and noise.

    Regularization :
      Do the same steps for a regularization batch, and add the error to the loss

    """
    #STEP 1 : INVERSION_LOSS

    latents = inversion_batch["pixel_values"]

    # pour chaque latent créé un bruit gaussien de même taille
    noise = torch.randn_like(latents)
    bsz = latents.shape[0]

    #on tire bsz (batch_size) valeurs entre 0 et le nombre d'étapes de bruitage du scheduler, pour que chaque latent soit bruité à un niveau différent
    timesteps = torch.randint(
        0,
        int(scheduler.config.num_train_timesteps*t_mutliplier),
        (bsz,),
        #torch.rand_like fait hériter le tenseur du device de latents donc on n'a pas besoin de l'expliciter contrairement à ici
        device=latents.device,
    )
    timesteps = timesteps.long()

    # le scheduler utilise la formule avec les alpha_t et les beta_t vue dans le modèle probabiliste de la diffusion pour passer de x0 directement à xt sans passer par les xt-1
    noisy_latents = scheduler.add_noise(latents, noise, timesteps)

    encoder_hidden_states = text_encoder(
        inversion_batch["input_ids"].to(text_encoder.device)
    )[0]

    inversion_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample

    inversion_target = noise

    # #Affichage pour contrôler
    # if global_step % control_step == 0 :
    #   idx = random.randint(0,bsz-1)
    #   latent = latents[idx]
    #   noisy_latent = noisy_latents[idx]
    #   print("Une image en entrainement à %s steps à l'étape de bruitage %s" % (global_step,timesteps[idx]))
    #   # 1. Remettre le scale factor
    #   latent = latent / 0.18215  # valeur utilisée dans SD 1.x
    #   img = TF.to_pil_image(latent)
    #   plt.imshow(img)
    #   plt.axis("off")
    #   plt.show()
    #   print("Version bruitée de cette image")
    #   latent = noisy_latent
    #   # 1. Remettre le scale factor
    #   latent = latent / 0.18215  # valeur utilisée dans SD 1.x
    #   img = TF.to_pil_image(latent)
    #   plt.imshow(img)
    #   plt.axis("off")
    #   plt.show()

    #STEP 2 : REGULARIZATION_LOSS

    if do_regularization :

      latents = regularization_batch["pixel_values"]

      # pour chaque latent créé un bruit gaussien de même taille
      noise = torch.randn_like(latents)

      bsz = latents.shape[0]

      #on tire bsz (batch_size) valeurs entre 0 et le nombre d'étapes de bruitage du scheduler, pour que chaque latent soit bruité à un niveau différent
      timesteps = torch.randint(
          0,
          int(scheduler.config.num_train_timesteps*t_mutliplier),
          (bsz,),
          #torch.rand_like fait hériter le tenseur du device de latents donc on n'a pas besoin de l'expliciter contrairement à ici
          device=latents.device,
      )
      timesteps = timesteps.long()

      # le scheduler utilise la formule avec les alpha_t et les beta_t vue dans le modèle probabiliste de la diffusion pour passer de x0 directement à xt sans passer par les xt-1
      noisy_latents = scheduler.add_noise(latents, noise, timesteps)


      encoder_hidden_states = text_encoder(
          regularization_batch["input_ids"].to(text_encoder.device)
      )[0]

      regularization_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample

      regularization_target = noise

    #STEP 3 : Calcul de la loss


    #moyenne de la différence au carré de chaque pixel pour chaque image du batch
    inversion_loss_per_image = ((inversion_pred - inversion_target) ** 2).mean(dim=[1, 2, 3])
    if do_regularization :
      regularization_loss_per_image = ((regularization_pred - regularization_target) ** 2).mean(dim=[1, 2, 3])

    #somme de l'erreur pour le tuning et de l'erreur pour le prior pour chaque image du batch
    if do_regularization :
      combined_loss_per_image = inversion_loss_per_image + 2*regularization_loss_per_image
    else :
      combined_loss_per_image = inversion_loss_per_image

    #moyenne sur le batch
    loss = combined_loss_per_image.mean()

    #Affichage d'une image originale et de sa version bruitée tous les control_step


    return loss


In [ ]:
def perform_inversion(
    unet,
    text_encoder,
    inversion_dataloader,
    num_steps: int,
    save_steps,
    control_step,
    scheduler,
    index_no_updates,
    optimizer,
    placeholder_token_ids,
    initializer_token_ids,
    placeholder_tokens,
    save_path: str,
    tokenizer,
    lr_scheduler,
    out_name : str,
    ti_lr : float
):

    """
    Trains the embeddings of the new tokens of the text_encoder and returns lists that allow to visualize the evolution of weights, gradients, and loss
    with training.

    """
    index_updates = ~index_no_updates
    ids_updates = []
    #on récupère les indexs à mettre à jour dans une nouvelle liste ids_updates pour contrôler que les ids sont bons
    for i in range (len(index_updates)) :
      if index_updates[i] :
        ids_updates.append(i)
    print("Vérification de l'égalité de index_updates %s et de placeholder_token_ids %s" % (ids_updates,placeholder_token_ids))

    #ces listes stockent l'évolution des grandeurs qu'on pourra afficher à la fin de l'exécution
    ti_loss = []
    ti_grad = []
    ti_dist = {}
    ti_sim = {}
    for i,j in zip(placeholder_token_ids,initializer_token_ids) :
      ti_dist[f"{tokenizer.convert_ids_to_tokens(i)}_{tokenizer.convert_ids_to_tokens(j)}"] = []
      ti_sim[f"{tokenizer.convert_ids_to_tokens(i)}_{tokenizer.convert_ids_to_tokens(j)}"] = []

    progress_bar = tqdm(range(num_steps))
    progress_bar.set_description("Steps")
    global_step = 0

    #.eval() = pas d’éléments stochastiques comme le dropout ou la batchnorm dynamique, les gradients sont quand même calculés
    unet.eval()
    text_encoder.train()

    #on ne peut pas passer une slice de .weight à l'optimizer donc on va tout mettre à jour puis corriger manuellement pour les tokens qu'on veut pas entrainer
    orig_embeds_params = text_encoder.get_input_embeddings().weight.data.clone()

    for epoch in range(math.ceil(num_steps / len(inversion_dataloader))):

        for batch in inversion_dataloader:

            #le scheduler est créé avec num_training_steps = num_epochs * len(dataloader)
            #il fait passer le lr à l'étape suivante
            #par exemple s'il suit un cosine : le LR commence haut, décroît doucement, et approche 0 de manière "cosine"
            #current_lr = lr_scheduler.get_last_lr()[0] pour voir l'évolution
            lr_scheduler.step()

            loss = (
                loss_step(
                    batch,
                    unet,
                    text_encoder,
                    scheduler,
                    control_step = control_step,
                    global_step = global_step
                )
            )
            loss.backward()

            ti_loss.append(loss.detach().item())
            # Tensor de shape [V, D]
            grads = text_encoder.get_input_embeddings().weight.grad
            #Calcule la norme L2 des gradients par vecteur, shape : [V]
            grad_norms = grads.norm(p=2, dim=1)
            #Moyenne des normes
            mean_grad_norm = grad_norms.mean().item()
            ti_grad.append(mean_grad_norm)
            optimizer.step()
            optimizer.zero_grad()
            with torch.no_grad() :
                #Suivi de la distance des nouveaux tokens aux tokens de départ
                for i,j in zip(placeholder_token_ids,initializer_token_ids) :
                  vec_i = text_encoder.get_input_embeddings().weight[i]
                  vec_j = text_encoder.get_input_embeddings().weight[j]
                  ti_dist[f"{tokenizer.convert_ids_to_tokens(i)}_{tokenizer.convert_ids_to_tokens(j)}"].append(torch.norm(vec_i - vec_j, p=2).item())
                  sim = cos_abs = torch.nn.functional.cosine_similarity(vec_i.unsqueeze(0),vec_j.unsqueeze(0)).item()
                  ti_sim[f"{tokenizer.convert_ids_to_tokens(i)}_{tokenizer.convert_ids_to_tokens(j)}"].append(sim)
                #récupération des embeddings d'origine pour les anciens tokens
                text_encoder.get_input_embeddings().weight[
                    index_no_updates
                ] = orig_embeds_params[index_no_updates]

            global_step += 1
            progress_bar.update(1)
            if global_step % save_steps == 0 :
                  save_all(
                    unet,
                    text_encoder,
                    placeholder_token_ids=placeholder_token_ids,
                    placeholder_tokens=placeholder_tokens,
                    save_ti = True,
                    save_lora = False,
                    save_path=os.path.join(save_path, f"inversion_{global_step}_{ti_lr}_{out_name}.safetensors"),
                )


    return ti_loss, ti_grad, ti_dist,ti_sim

In [ ]:
def perform_tuning(
    unet,
    text_encoder,
    inversion_dataloader,
    regularization_dataloader,
    num_steps,
    save_steps,
    scheduler,
    optimizer,
    placeholder_token_ids,
    placeholder_tokens,
    save_path,
    lr_scheduler_lora,
    lora_unet_target_modules,
    lora_clip_target_modules,
    tokenizer,
    tracked_lora_paths,tracked_lora_weights,tracked_lora_grads,
    do_regularization : Optional[bool] = False,
    out_name : Optional[str] = None,
    unet_lr : Optional[float] = 1e-4
):
    """
    Trains the LORAs placed in the UNET and the text encoder and returns lists allowing to view the evolution of weights, gradients, and loss
    with training.

    """
    progress_bar = tqdm(range(num_steps))
    progress_bar.set_description("Steps")
    global_step = 0

    tracked_tuning_loss = []
    tracked_tuning_grad = []


    unet.train()
    text_encoder.train()

    for epoch in range(math.ceil(num_steps / len(inversion_dataloader))):
        for i, (inversion_batch, regularization_batch) in enumerate (zip(inversion_dataloader,regularization_dataloader)):
            if epoch == 0 and i == 0 :
              print("Verification que le token cible a bien été remplacé %s" % tokenizer.convert_ids_to_tokens(inversion_batch["input_ids"][0]))

            tuning_loss = loss_step(
                inversion_batch,
                unet,
                text_encoder,
                scheduler,
                t_mutliplier=0.8,
                regularization_batch=regularization_batch,
                do_regularization=do_regularization
            )

            tuning_loss.backward()

            #ajout des valeurs importantes aux listes pour surveiller l'évolution
            tracked_tuning_loss.append(tuning_loss.detach().item())
            add_tracked_lora_weights(unet, tracked_lora_paths,tracked_lora_weights,tracked_lora_grads)
            # add_tracked_lora_weights(text_encoder, tracked_lora_paths,tracked_lora_weights,tracked_lora_grads)

            optimizer.step()
            optimizer.zero_grad()

            progress_bar.update(1)
            logs = {
                "loss": tuning_loss.detach().item(),
                "lr": lr_scheduler_lora.get_last_lr()[0],
            }
            progress_bar.set_postfix(**logs)

            global_step += 1
            if global_step % save_steps == 0 :
              save_all(
                  unet,
                  text_encoder,
                  placeholder_token_ids=placeholder_token_ids,
                  placeholder_tokens=placeholder_tokens,
                  save_path=os.path.join(save_path, f"tuning_{save_steps}_{unet_lr}_{out_name}.safetensors"),
                  save_ti = False,
                  save_lora = True
              )
    return tracked_tuning_loss

In [ ]:
def train(
    placeholder_tokens,
    initializer_tokens,
    placeholder_token_ids,
    initializer_token_ids,
    text_encoder,
    unet,
    tokenizer,
    noise_scheduler,
    index_no_updates,
    inversion_dataloader,
    regularization_dataloader : Optional[torch.utils.data.DataLoader],
    max_train_steps_tuning: int = 0,
    max_train_steps_ti: int = 5000,
    save_steps : int = 1000,
    control_step : int = 100,
    lora_rank: int = 4,
    lora_unet_target_modules={"UNet2DConditionModel"},
    lora_clip_target_modules={},
    learning_rate_unet: float = 1e-4,
    learning_rate_text: float = 1e-5,
    learning_rate_ti: float = 5e-4,
    lr_scheduler: str = "linear",
    lr_scheduler_lora: str = "linear",
    device="cuda:0",
    do_inversion : Optional[bool] = True,
    do_pivotal_tuning : Optional[bool] = False,
    do_regularization : Optional[bool] = False,
    out_name : Optional[str] = None,
    output_dir : Optional[str] = None,
):

    """
    Builds the scheduler and optimizer to execute perform_inversion.
    Injects the loras into the models at the designated locations.
    Builds the scheduler and optimizer to execute perform_tuning.

    """
    unet_lr = learning_rate_unet
    text_encoder_lr = learning_rate_text
    ti_lr = learning_rate_ti

    # STEP 1 : Perform Inversion

    if do_inversion :

      unet.requires_grad_(False)
      text_encoder.requires_grad_(True)

      #text_encoder = CLIPTextModel
      #text_encoder à un attribut text_model qui est un CLIPTextTransformer
      #CLIPTextTransformer a des attributs embeddings = CLIPTextEmbeddings(config), encoder = CLIPEncoder(config) et final_layer_norm = nn.LayerNorm(embed_dim, eps=config.layer_norm_eps)
      #CLIPTextEmbeddings a deux attributs token_embedding et position_embedding qui sont des nn.Embedding
      #get_input_embeddings renvoie self.text_model.embeddings.token_embedding
      #https://github.com/huggingface/transformers/blob/main/src/transformers/models/clip/modeling_clip.py#L48
      #on freeze tout sauf les embeddings des tokens
      params_to_freeze = itertools.chain(
          text_encoder.text_model.encoder.parameters(),
          text_encoder.text_model.final_layer_norm.parameters(),
          text_encoder.text_model.embeddings.position_embedding.parameters(),
      )
      for param in params_to_freeze:
          param.requires_grad = False

      ti_optimizer = optim.AdamW(
          #tous les embeddings sont passés à l'optimizer mais seuls ceux dont a calculé le gradient donc les nouveaux ids seront mis à jour
          text_encoder.get_input_embeddings().parameters(),
          lr=ti_lr
      )

      #https://huggingface.co/docs/transformers/main_classes/optimizer_schedules
      #applique les schedulers de cette page comme get_cosine_schedule_with_warmup
      #le warm-up est une phase pendant laquelle lr augmente linéairement de 0 au lr choisi dans l'optimizer
      lr_scheduler = get_scheduler(
          lr_scheduler,
          optimizer=ti_optimizer,
          num_training_steps=max_train_steps_ti,
          num_warmup_steps = 0
      )

      ti_loss,ti_grad,ti_weights,ti_sim = perform_inversion(
          unet,
          text_encoder,
          inversion_dataloader,
          max_train_steps_ti,
          save_steps = save_steps,
          control_step = control_step,
          scheduler=noise_scheduler,
          index_no_updates=index_no_updates,
          optimizer=ti_optimizer,
          lr_scheduler=lr_scheduler,
          placeholder_tokens=placeholder_tokens,
          placeholder_token_ids=placeholder_token_ids,
          initializer_token_ids=initializer_token_ids,
          save_path=output_dir,
          tokenizer=tokenizer,
          out_name = out_name,
          ti_lr=ti_lr)

      del ti_optimizer


    # Next perform Tuning with LoRA

    if do_pivotal_tuning :

      #ces listes vont permettre d'étudier l'évolution des loras avec l'entrainement
      tracked_lora_paths = {"UNet2DConditionModel":[]}
      tracked_lora_weights = {}
      tracked_lora_grads = {}

      unet.requires_grad_(False)
      text_encoder.requires_grad_(False)

      #les paramètres des loras injectés sont mis à requires_grad = True dans cette fonction
      unet_lora_params, _ = inject_trainable_lora_extended(
          unet, r=lora_rank, target_replace_module=lora_unet_target_modules,tracked_lora_paths=tracked_lora_paths,tracked_lora_weights=tracked_lora_weights,tracked_lora_grads=tracked_lora_grads
      )
      print(f"Injection de {len(unet_lora_params)} loras")

      #si on modifie le code et qu'on veut réexécuter train, inject_trainable_lora_extended ne va détecter aucun module linear ou conv car ils ont déjà
      #tous été remplacés par des loras. Toutefois il faut quand même remettre les paramètres des loras précédemment injectés à requires_grad = True
      #et les ajouter à la liste unet_lora_params ce que fait cette fonction
      if len(unet_lora_params) == 0:
          unet_lora_params = set_lora_grad(unet, r=lora_rank, target_replace_module=lora_unet_target_modules,tracked_lora_paths=tracked_lora_paths,tracked_lora_weights=tracked_lora_weights,tracked_lora_grads=tracked_lora_grads)

      print(f"{len(unet_lora_params)} loras mis à requires_grad = True")
      #unet_lora_params est une liste de générateurs de paramètres pour les loras up et down
      #ex : lora_up.parameters() = <generator object Module.parameters at 0x794984cad7e0>
      #itertools.chain.from_iterable parcourt les générateurs de la liste et retourne leur contenu à la suite
      #par exemple si on a deux lora_up.parameters) dans la liste, on obtient : lora_up1.weight,lora_up2.weight
      params_to_optimize = [
          {"params": itertools.chain.from_iterable(unet_lora_params), "lr": unet_lr},
      ]

      # text_encoder_lora_params, _ = inject_trainable_lora_extended(
      #     text_encoder,
      #     r=lora_rank,
      #     target_replace_module=lora_clip_target_modules,
      #     tracked_lora_paths=tracked_lora_paths,tracked_lora_weights=tracked_lora_weights,tracked_lora_grads=tracked_lora_grads
      # )

      # if len(text_encoder_lora_params) == 0:
      #     text_encoder_lora_params = set_lora_grad(text_encoder, r=lora_rank, target_replace_module=lora_clip_target_modules,tracked_lora_paths=tracked_lora_paths,tracked_lora_weights=tracked_lora_weights,tracked_lora_grads=tracked_lora_grads)

      # params_to_optimize += [
      #     {
      #         "params": itertools.chain.from_iterable(text_encoder_lora_params),
      #         "lr": text_encoder_lr,
      #     }
      # ]
      lora_optimizers = optim.AdamW(params_to_optimize)

      lr_scheduler_lora = get_scheduler(
          lr_scheduler_lora,
          optimizer=lora_optimizers,
          num_training_steps=max_train_steps_tuning,
          num_warmup_steps = 0
      )

      if do_regularization :
        tracked_tuning_loss = perform_tuning(
            unet,
            text_encoder,
            inversion_dataloader,
            regularization_dataloader,
            max_train_steps_tuning,
            save_steps,
            scheduler=noise_scheduler,
            optimizer=lora_optimizers,
            placeholder_tokens=placeholder_tokens,
            placeholder_token_ids=placeholder_token_ids,
            save_path=output_dir,
            lr_scheduler_lora=lr_scheduler_lora,
            lora_unet_target_modules=lora_unet_target_modules,
            lora_clip_target_modules=lora_clip_target_modules,
            tokenizer=tokenizer,
            tracked_lora_paths=tracked_lora_paths,tracked_lora_weights=tracked_lora_weights,tracked_lora_grads=tracked_lora_grads,
            do_regularization = True,
            out_name = out_name,
            unet_lr = unet_lr
        )
      else :
            tracked_tuning_loss = perform_tuning(
            unet,
            text_encoder,
            inversion_dataloader,
            inversion_dataloader,
            max_train_steps_tuning,
            save_steps,
            scheduler=noise_scheduler,
            optimizer=lora_optimizers,
            placeholder_tokens=placeholder_tokens,
            placeholder_token_ids=placeholder_token_ids,
            save_path=output_dir,
            lr_scheduler_lora=lr_scheduler_lora,
            lora_unet_target_modules=lora_unet_target_modules,
            lora_clip_target_modules=lora_clip_target_modules,
            tokenizer=tokenizer,
            tracked_lora_paths=tracked_lora_paths,tracked_lora_weights=tracked_lora_weights,tracked_lora_grads=tracked_lora_grads,
            do_regularization = False,
            out_name = out_name,
            unet_lr =unet_lr
        )

    if do_pivotal_tuning :
      return tracked_tuning_loss, tracked_lora_grads,tracked_lora_weights,tracked_lora_paths
    else :
      return ti_loss,ti_grad,ti_weights,ti_sim

In [ ]:
"""
Loras classes, functions to inject them and activate their gradients

"""

In [ ]:
class LoraInjectedLinear(nn.Module):
    def __init__(
        self, in_features, out_features, bias=False, r=4, dropout_p=0.1, scale=1.0
    ):
        """
        Module consisting of a linear module with in and out features and a lora.
        set_selector_from_diag allows you to place a diagonal matrix between the lora down and the lora up.
        """
        super().__init__()

        if r > min(in_features, out_features):
            raise ValueError(
                f"LoRA rank {r} must be less or equal than {min(in_features, out_features)}"
            )
        self.r = r
        self.linear = nn.Linear(in_features, out_features, bias)
        self.lora_down = nn.Linear(in_features, r, bias=False)
        self.dropout = nn.Dropout(dropout_p)
        self.lora_up = nn.Linear(r, out_features, bias=False)
        self.scale = scale
        self.selector = nn.Identity()

        #initialisation des paramètres avec une loi normale de moyenne 0 et d'écart-type 1/r
        nn.init.normal_(self.lora_down.weight, std=1 / r)
        nn.init.zeros_(self.lora_up.weight)

        # Freeze all params by default
        for param in self.parameters():
            param.requires_grad = False

        # Unfreeze only LoRA parameters
        self.lora_up.weight.requires_grad = True
        self.lora_down.weight.requires_grad = True


    def forward(self, input):
        return (
            self.linear(input)
            #drop randomly zeroes some of the elements of the input tensor with probability p during training
            + self.dropout(self.lora_up(self.selector(self.lora_down(input))))
            * self.scale
        )

    def realize_as_lora(self):
        return self.lora_up.weight.data * self.scale, self.lora_down.weight.data

    def set_selector_from_diag(self, diag: torch.Tensor):
        # diag is a 1D tensor of size (r,)
        assert diag.shape == (self.r,)
        self.selector = nn.Linear(self.r, self.r, bias=False)
        self.selector.weight.data = torch.diag(diag)
        self.selector.weight.data = self.selector.weight.data.to(
            self.lora_up.weight.device
        ).to(self.lora_up.weight.dtype)

In [ ]:
class LoraInjectedConv2d(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size,
        stride=1,
        padding=0,
        dilation=1,
        groups: int = 1,
        bias: bool = True,
        r: int = 4,
        dropout_p: float = 0.1,
        scale: float = 1.0,
    ):
        """
        Module consisting of a conv2d module with in and out features and a lora.
        Here, the diagonal matrix is a 1d convolution and not a linear one because the input is in 3d with multiple channels.
        """
        super().__init__()
        if r > min(in_channels, out_channels):
            raise ValueError(
                f"LoRA rank {r} must be less or equal than {min(in_channels, out_channels)}"
            )
        self.r = r
        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
            groups=groups,
            bias=bias,
        )

        self.lora_down = nn.Conv2d(
            in_channels=in_channels,
            out_channels=r,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
            groups=groups,
            bias=False,
        )
        self.dropout = nn.Dropout(dropout_p)
        self.lora_up = nn.Conv2d(
            in_channels=r,
            out_channels=out_channels,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False,
        )
        self.selector = nn.Identity()
        self.scale = scale

        nn.init.normal_(self.lora_down.weight, std=1 / r)
        nn.init.zeros_(self.lora_up.weight)

        # Freeze all params by default
        for param in self.parameters():
            param.requires_grad = False

        # Unfreeze only LoRA parameters
        self.lora_up.weight.requires_grad = True
        self.lora_down.weight.requires_grad = True

    def forward(self, input):
        return (
            self.conv(input)
            + self.dropout(self.lora_up(self.selector(self.lora_down(input))))
            * self.scale
        )

    def realize_as_lora(self):
        return self.lora_up.weight.data * self.scale, self.lora_down.weight.data

    def set_selector_from_diag(self, diag: torch.Tensor):
        # diag is a 1D tensor of size (r,)
        assert diag.shape == (self.r,)
        self.selector = nn.Conv2d(
            in_channels=self.r,
            out_channels=self.r,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False,
        )
        self.selector.weight.data = torch.diag(diag)

        # same device + dtype as lora_up
        self.selector.weight.data = self.selector.weight.data.to(
            self.lora_up.weight.device
        ).to(self.lora_up.weight.dtype)

In [ ]:
def _find_modules(
    model,
    ancestor_class: Optional[Set[str]] = None,
    search_class: List[Type[nn.Module]] = [nn.Linear],
    exclude_children_of: Optional[List[Type[nn.Module]]] = [
        LoraInjectedLinear,
        LoraInjectedConv2d,
    ],
    tracked_lora_paths: Optional[Dict[str,List[str]]] = None,
    tracked_lora_weights : Optional[Dict[str,List[float]]] = None,
    tracked_lora_grads : Optional[Dict[str,List[float]]] = None
):
    """
    Find all modules of a certain class (or union of classes) that are direct or
    indirect descendants of other modules of a certain class (or union of classes).

    Returns all matching modules, along with the parent of those modules and the
    names they are referenced by.
    """

    # Get the targets we should replace all linears under
    if ancestor_class is not None:
        ancestors = [
            [fullname, module]
            for fullname, module in model.named_modules()
            if module.__class__.__name__ in ancestor_class
        ]
    else:
        # this, incase you want to naively iterate over all modules.
        ancestors = [module for module in model.modules()]

    # For each target find every linear_class module that isn't a child of a LoraInjectedLinear
    for ancestor_fullname,ancestor in ancestors:
        #.named_modules() donne le chemin entier de l'ancestor au module
        for fullname, module in ancestor.named_modules():
            if any([isinstance(module, _class) for _class in search_class]):
                # Find the direct parent if this is a descendant, not a child, of target
                #"encoder.layer.2.attention.linear" devient path = ['encoder', 'layer', '2', 'attention'] et name = 'linear'
                *path, name = fullname.split(".")
                parent = ancestor
                while path:
                    #path.pop(0) = encoder
                    #parent.get_submodule(path.pop(0)) = parent.encoder
                    #au final : parent.encoder.layer.2.attention
                    parent = parent.get_submodule(path.pop(0))
                # Pour ne pas injecter des loras dans des loras si par exemple on exécute plusieurs fois train
                if exclude_children_of and any(
                    [isinstance(parent, _class) for _class in exclude_children_of]
                ):
                    continue
                #on met à jour les listes qui stockent les valeurs de l'entrainement, en utilisant le chemin complet jusqu'au module où le lora
                #va être injecté
                full_path = f"{model.__class__.__name__}.{ancestor_fullname}.{fullname}" if ancestor_fullname else fullname
                if tracked_lora_paths is not None:
                  tracked_lora_paths[model.__class__.__name__].append(full_path)
                  tracked_lora_weights[full_path] = []
                  tracked_lora_grads[full_path] = []
                # Otherwise, yield it
                yield full_path, parent, name, module

In [ ]:
def inject_trainable_lora_extended(
    model: nn.Module,
    target_replace_module: Set[str] = UNET_TARGET_REPLACE,
    r: int = 4,
    loras=None,  # path to lora .pt
    tracked_lora_paths: Optional[Dict[str,List[str]]] = None,
    tracked_lora_weights : Optional[Dict[str,List[float]]] = None,
    tracked_lora_grads: Optional[Dict[str,List[float]]] = None
):
    """
    Returns require_grad_params, the list of parameters for the injected loras, as well as the list names of the names of the modules where the loras were injected.


    Parameters:
        loras:
            list of lora parameters if the weights to be assigned to the injected loras are already known
    Output:
        require_grad_params:
            e.g.: [lora1_up_params,lora1_down_params,lora2_up_params,lora2_down_params]
    """

    require_grad_params = []
    names = []

    if loras != None:
        loras = torch.load(loras)

    for _,_module, name, _child_module in _find_modules(
        model, target_replace_module, search_class=[nn.Linear, nn.Conv2d],tracked_lora_paths=tracked_lora_paths,tracked_lora_weights=tracked_lora_weights,tracked_lora_grads=tracked_lora_grads
    ):
        if _child_module.__class__ == nn.Linear:
            weight = _child_module.weight
            bias = _child_module.bias
            _tmp = LoraInjectedLinear(
                _child_module.in_features,
                _child_module.out_features,
                _child_module.bias is not None,
                r=r,
            )
            _tmp.linear.weight = weight
            if bias is not None:
                _tmp.linear.bias = bias
        elif _child_module.__class__ == nn.Conv2d:
            weight = _child_module.weight
            bias = _child_module.bias
            _tmp = LoraInjectedConv2d(
                _child_module.in_channels,
                _child_module.out_channels,
                _child_module.kernel_size,
                _child_module.stride,
                _child_module.padding,
                _child_module.dilation,
                _child_module.groups,
                _child_module.bias is not None,
                r=r,
            )

            _tmp.conv.weight = weight
            if bias is not None:
                _tmp.conv.bias = bias

        # switch the module
        _tmp.to(_child_module.weight.device).to(_child_module.weight.dtype)
        if bias is not None:
            _tmp.to(_child_module.bias.device).to(_child_module.bias.dtype)

        _module._modules[name] = _tmp

        require_grad_params.append(_module._modules[name].lora_up.parameters())
        require_grad_params.append(_module._modules[name].lora_down.parameters())

        if loras != None:
            _module._modules[name].lora_up.weight = loras.pop(0)
            _module._modules[name].lora_down.weight = loras.pop(0)

        names.append(name)

    return require_grad_params, names

In [ ]:
def set_lora_grad(
    model: nn.Module,
    target_replace_module: Set[str] = UNET_TARGET_REPLACE,
    r: int = 4,
    loras=None,  # path to lora .pt
    tracked_lora_paths: Optional[Dict[str,List[str]]] = None,
    tracked_lora_weights : Optional[Dict[str,List[float]]] = None,
    tracked_lora_grads: Optional[Dict[str,List[float]]] = None
):
    """
    Activates loras grad of all loras injected in target_replace_module. Useful when training failed after injecting, and you just want to reactivate
    loras without going through the injection phase.

    """

    require_grad_params = []

    for _,_module, name, _child_module in _find_modules(
        model, target_replace_module, search_class=[LoraInjectedLinear, LoraInjectedConv2d],exclude_children_of=[],tracked_lora_paths=tracked_lora_paths,tracked_lora_weights=tracked_lora_weights,tracked_lora_grads=tracked_lora_grads
    ):
        _child_module.lora_up.weight.requires_grad = True
        _child_module.lora_down.weight.requires_grad = True
        require_grad_params.append(_child_module.lora_up.parameters())
        require_grad_params.append(_child_module.lora_down.parameters())

    return require_grad_params

In [ ]:
def add_tracked_lora_weights(model, tracked_paths,tracked_lora_weights,tracked_lora_grads):
    """
    Adds the current weights and gradients of the LORAs to the tracked_lora_weights and tracked_lora_grads lists, knowing the paths of the LORAs.

    """
    for path in tracked_paths[model.__class__.__name__]:
        module = model
        for key in path.split("."):
            if key != model.__class__.__name__ :
              module = getattr(module, key)
        with torch.no_grad() :
          if hasattr(module, "lora_up") and hasattr(module, "lora_down"):
              up = module.lora_up
              down = module.lora_down
              if isinstance(up, nn.Linear):
                  delta = (up.weight @ down.weight).flatten().abs().mean().item()
              else:
                  delta = (up.weight.flatten().abs().mean() * down.weight.flatten().abs().mean()).item()
              tracked_lora_weights[path].append(delta)

              # Suivi des gradients
              grad_up = module.lora_up.weight.grad
              grad_down = module.lora_down.weight.grad

              if grad_up is not None and grad_down is not None:
                  norm_up = grad_up.norm().item()
                  norm_down = grad_down.norm().item()
                  grad_norm = (norm_up + norm_down) / 2  # moyenne des deux
              else:
                  grad_norm = 0.0  # ou None si tu veux détecter les absents

              tracked_lora_grads[path].append(grad_norm)


In [ ]:
"""
This section defines the functions used to save a configuration in safetensor and load it.

"""

In [ ]:
"""
Roadmap for saving LORAs and text embeddings

1°) extract_lora_as_tensor:
    Returns a list of LORAs containing tuples (up,down) of all LORAs that have been injected into the model at target_replace_module

2°) save_safeloras_with_embeds:
    Builds a safetensor with :
      - a weight dictionary of LORA weights retrieved with extract_lora_as_tensor, and text embeddings
      - a metadata dictionary containing additional information, including the rank of each LORA.

3°) save_all:
    Builds two dictionaries, modelmaps and embeds, to be passed as arguments to save_safeloras_with_embeds.
"""


In [ ]:
def extract_lora_as_tensor(
    model, target_replace_module, as_fp16=True
):

    """
    Returns a list of loras containing tuples (up,down) of all the loras that have been injected into the model at target_replace_module.

    Parameters:
        model (`nn.Module`):
            the unet or text_encoder model
        target_replace_module (‘set[str]’):
            the modules (Transformer2DModel,CLIPTextTransformer) where the linear and conv loras have been placed

    Output:
        loras: list of tuples (up,down) of the weights of each lora


    """

    loras = []

    for _,_m, _n, _child_module in _find_modules(
        model,
        target_replace_module,
        search_class=[LoraInjectedLinear, LoraInjectedConv2d],
    ):
        #realize_as_lora renvoie les poids scale*up,down du lora
        up, down = _child_module.realize_as_lora()
        if as_fp16:
            up = up.to(torch.float16)
            down = down.to(torch.float16)

        loras.append((up, down))

    if len(loras) == 0:
        raise ValueError("No lora injected.")

    return loras

In [ ]:
def save_safeloras_with_embeds(
    modelmap: Dict[str, Tuple[nn.Module, Set[str]]] = {},
    embeds: Dict[str, torch.Tensor] = {},
    outpath="./lora.safetensors",
):
    """
    Parameters:
        modelmap (‘Dict[str, Tuple[nn.Module, Set[str]]]’) :
            dictionary with the model name, the model, and the target modules, e.g., “unet”: (unet,{“Transformer2DModel”})
        embeds (‘Dict[str, torch.Tensor]’):
            dictionary with the embedding ID and the corresponding tensor for new tokens
        outpath (‘String’):
            the save path
    Output:
        safetensor:
            a weights dictionary with the weight of all loras and the weight of all embeddings
            e.g.: “unet:1:up” for the up weights of the first lora of the unet
            e.g.: “<tok1>” for the weight of the embedding of token 1
            a metadata dictionary with additional information
            e.g.: “unet:1:rank” for the rank of the first lora of the unet
    """

    weights = {}
    metadata = {}

    for name, (model, target_replace_module) in modelmap.items():
        metadata[name] = json.dumps(list(target_replace_module))

        for i, (_up, _down) in enumerate(
            extract_lora_as_tensor(model, target_replace_module)
        ):
            rank = _down.shape[0]

            metadata[f"{name}:{i}:rank"] = str(rank)
            weights[f"{name}:{i}:up"] = _up
            weights[f"{name}:{i}:down"] = _down

    for token, tensor in embeds.items():
        metadata[token] = EMBED_FLAG
        weights[token] = tensor

    print(f"Saving weights to {outpath}")
    safe_save(weights, outpath, metadata)

In [ ]:
def save_all(
    unet,
    text_encoder,
    save_path,
    placeholder_token_ids=None,
    placeholder_tokens=None,
    save_ti=True,
    save_lora=True,
    target_replace_module_text=TEXT_ENCODER_TARGET_REPLACE,
    target_replace_module_unet=UNET_TARGET_REPLACE,
    safe_form=True,
):
    """
    Builds the modelmaps and embeds dictionaries to pass as arguments to save_safeloras_with_embeds

    """

    loras = {}
    embeds = {}

    if save_lora:
        loras["unet"] = (unet, target_replace_module_unet)
        # loras["text_encoder"] = (text_encoder, target_replace_module_text)

    if save_ti:
        for tok, tok_id in zip(placeholder_tokens, placeholder_token_ids):
            learned_embeds = text_encoder.get_input_embeddings().weight[tok_id]
            embeds[tok] = learned_embeds.detach().cpu()

    save_safeloras_with_embeds(loras, embeds, save_path)

In [ ]:
"""
Roadmap for loading LORAs from Safetensor

1°) parse_safeloras:
    Rebuilds a usable LORA dictionary of weights and ranks from Safetensor containing the weights and metadata dictionaries.

2°) apply_lora_on_model:
    Adds the weights of the LORAs from the LORAs list to all modules of the model

3°) apply_lora_from_safetensor:
    Builds the LORAs dictionary from the safetensor with parse_safeloras, applies this dictionary to the UNET and text_encoder with
    apply_lora_on_model
"""

In [ ]:
def parse_safeloras(
    safeloras,
) -> Dict[str, Tuple[List[nn.parameter.Parameter], List[int], List[str]]]:
    """
    Rebuilds a usable loras dictionary of weights and ranks
    from the safetensor containing the weights and metadata dictionaries.
    The loras dictionary is as follows:
    - lora[model] where model is either “unet” or “text_encoder,” is a tuple (weights, ranks, target)
    - weights is the list of up and down weights of all the model's loras
    - ranks are the ranks of the loras
    - target are the targeted modules

    """
    loras = {}
    #safeloras.metadata() récupère le dictionnaire metadate et safeloras.keys() récupère les clés du dictionnaire weights uniquement
    metadata = safeloras.metadata()
    #donne le modèle de la clé ex : "unet:1:up" -> "unet"
    get_name = lambda k: k.split(":")[0]

    keys = list(safeloras.keys())
    #le tri se fait sur le modèle : unet, text_encoder ou les tokens comme <tok1>,<tok2>. Ici on a ajouté toutes les clés unet
    #puis toutes les clés text_encoder dans extract_lora_as_tensor donc ce tri n'est pas forcément utile
    keys.sort(key=get_name)

    for name, module_keys in groupby(keys, get_name):
        info = metadata.get(name)

        if not info:
            raise ValueError(
                f"Tensor {name} has no metadata - is this a Lora safetensor?"
            )

        # Skip Textual Inversion embeds
        if info == EMBED_FLAG:
            continue

        # Handle Loras
        # Extract the targets
        target = json.loads(info)

        # Build the result lists - Python needs us to preallocate lists to insert into them
        module_keys = list(module_keys)
        ranks = [4] * (len(module_keys) // 2)
        weights = [None] * len(module_keys)

        for key in module_keys:
            # Split the model name and index out of the key
            _, idx, direction = key.split(":")
            idx = int(idx)

            # Add the rank
            ranks[idx] = int(metadata[f"{name}:{idx}:rank"])

            # Insert the weight into the list
            idx = idx * 2 + (1 if direction == "down" else 0)
            weights[idx] = nn.parameter.Parameter(safeloras.get_tensor(key))

        loras[name] = (weights, ranks, target)

    return loras

In [ ]:
def apply_lora_on_model(
    model,
    loras,
    target_replace_module,
    r: Union[int, List[int]] = 4,
    delta_weights : Dict[str, float] = {},
):
    """
    Adds the weight of the loras from the loras list to all modules in the model.

    """
    alpha = 1.0

    for full_path,_module, name, _child_module in _find_modules(
        model,
        target_replace_module,
        search_class=[nn.Linear, nn.Conv2d,]
    ):
        with torch.no_grad() :
          if (_child_module.__class__ == nn.Linear) :
              #La matrice de poids up ou down doit être de dimension 2 pour un linear
              if len(loras[0].shape) != 2:
                  continue
              up_weight = loras.pop(0)
              down_weight = loras.pop(0)
              delta = alpha*(up_weight @ down_weight)
              start_weight = _child_module.weight.detach().clone()
              _child_module.weight += delta.type(_child_module.weight.dtype).to(_child_module.weight.device)
              delta_weights[full_path] = torch.norm((_child_module.weight - start_weight),p='fro').item()
          elif (_child_module.__class__ == nn.Conv2d):
              #La matrice de poids up ou down doit être de dimension 4 pour un conv2d (out_channels, in_channels, kernel_height, kernel_width)
              #on a un filtre différent pour chaque duo out_channel,in_channel
              if len(loras[0].shape) != 4:
                  continue
              up_weight = loras.pop(0)
              down_weight = loras.pop(0)
              delta = alpha*(up_weight.flatten(start_dim=1) @ down_weight.flatten(start_dim=1)).reshape(_child_module.weight.data.shape)
              start_weight = _child_module.weight.detach().clone()
              _child_module.weight += delta.type(_child_module.weight.dtype).to(_child_module.weight.device)
              delta_weights[full_path] = torch.norm((_child_module.weight - start_weight),p='fro').item()

In [ ]:
def apply_lora_from_safetensor(pipe, safeloras):
    """
    Applies all safetensor loras to unet and text_encoder
    """
    loras = parse_safeloras(safeloras)
    #Dictionnary to store the norm of the differences of weight between the original model
    #and after loras are loaded, to ensure it worked
    delta_weights = {}

    for name, (lora, ranks, target) in loras.items():
        model = getattr(pipe, name, None)

        if not model:
            print(f"No model provided for {name}, contained in Lora")
            continue

        apply_lora_on_model(model, lora, target, ranks,delta_weights)
    return delta_weights

In [ ]:
"""
Roadmap for loading embeddings from safetensor

1°) parse_safeloras_embeds:
    Rebuilds a usable embed dictionary from safetensor containing the weights and metadata dictionaries.

2°) apply_learned_embed_in_clip:
    Uses the embed dictionary of new tokens to update their embedding.
"""

In [ ]:
def parse_safeloras_embeds(
    safeloras,
) -> Dict[str, torch.Tensor]:
    """
    Rebuilds a usable embeds dictionary of embeddings
    from safetensor with the weights and metadata dictionaries.
    The embeds dictionary is such that:
    - embeds[token] = embedding of the token

    """
    embeds = {}
    metadata = safeloras.metadata()

    for key in safeloras.keys():
        # Only handle Textual Inversion embeds
        meta = metadata.get(key)
        if not meta or meta != EMBED_FLAG:
            continue

        embeds[key] = safeloras.get_tensor(key)

    return embeds

In [ ]:
def apply_learned_embed_in_clip(
    learned_embeds,
    text_encoder,
    tokenizer,
    idempotent=False,
):

    """
    Add the tokens that are the keys of learned_embeds to the tokenizer, create their IDs, and add the embedding to the text_encoder.

    Parameters:
        idempotent:
            If the token already exists in the tokenizer, we will try to add it in another form, e.g.:
            “mytoken”, “mytoke-1>”, then “mytoke-2>”

    """

    trained_tokens = list(learned_embeds.keys())

    for token in trained_tokens:
        embeds = learned_embeds[token]

        # cast to dtype of text_encoder
        dtype = text_encoder.get_input_embeddings().weight.dtype
        num_added_tokens = tokenizer.add_tokens(token)

        i = 1
        if not idempotent:
            while num_added_tokens == 0:
                print(f"The tokenizer already contains the token {token}.")
                token = f"{token[:-1]}-{i}>"
                print(f"Attempting to add the token {token}.")
                num_added_tokens = tokenizer.add_tokens(token)
                i += 1
        elif num_added_tokens == 0 and idempotent:
            print(f"The tokenizer already contains the token {token}.")
            print(f"Replacing {token} embedding.")
        else :
          print(f"Ajout du token {token}")

        # resize the token embeddings
        text_encoder.resize_token_embeddings(len(tokenizer))

        # get the id for the token and assign the embeds
        token_id = tokenizer.convert_tokens_to_ids(token)
        text_encoder.get_input_embeddings().weight.data[token_id] = embeds
    return token

In [ ]:
def patch_pipe(
    pipe,
    maybe_unet_path,
    token: Optional[str] = None,
    r: int = 4,
    patch_unet=False,
    patch_text=False,
    patch_ti=True,
    idempotent_token=True,
    unet_target_replace_module=UNET_TARGET_REPLACE,
    text_target_replace_module=TEXT_ENCODER_TARGET_REPLACE,
):

    """
    Applies the LORAs to UNET and the text_encoder, as well as the embeddings of the new tokens to the tokenizer.

    """

    #framework="pt"	on veut accéder aux tensors au format PyTorch (torch.Tensor)
    safeloras = safe_open(maybe_unet_path, framework="pt", device="cpu")
    delta_weights = apply_lora_from_safetensor(pipe, safeloras)
    tok_dict = parse_safeloras_embeds(safeloras)
    if patch_ti:
        apply_learned_embed_in_clip(
            tok_dict,
            pipe.text_encoder,
            pipe.tokenizer,
        )
    return delta_weights

In [ ]:
"""
Functions for plotting graphs

"""

In [ ]:
def plot_unet_lora_grad_norms(tracked_lora_grads, steps, title="Gradient Norms of LoRAs over Steps for unet"):
    plt.figure(figsize=(12, 6))

    i = 0
    for lora_name, grad_norms in tracked_lora_grads.items():
        if i < 282 :
            plt.plot(steps, grad_norms, label=lora_name)
        i+=1

    plt.xlabel("Training Step")
    plt.ylabel("Gradient Norm")
    plt.title(title)
    plt.legend(loc="upper right", fontsize="small", bbox_to_anchor=(1.15, 1))
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
def plot_text_encoder_lora_grad_norms(tracked_lora_grads, steps, title="Gradient Norms of LoRAs over Steps for text_encoder"):
    plt.figure(figsize=(12, 6))

    i = 0
    for lora_name, grad_norms in tracked_lora_grads.items():
        if i >= 282 :
            plt.plot(steps, grad_norms, label=lora_name)
        i+=1

    plt.xlabel("Training Step")
    plt.ylabel("Gradient Norm")
    plt.title(title)
    plt.legend(loc="upper right", fontsize="small", bbox_to_anchor=(1.15, 1))
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_unet_lora_weights(tracked_lora_weights, steps, title="Weights Norms of LoRAs over Steps for unet"):
    plt.figure(figsize=(12, 6))

    i = 0
    for lora_name, grad_norms in tracked_lora_weights.items():
        if i < 282 :
            plt.plot(steps, grad_norms, label=lora_name)
        i+=1

    plt.xlabel("Training Step")
    plt.ylabel("Weights Norm")
    plt.title(title)
    plt.legend(loc="upper right", fontsize="small", bbox_to_anchor=(1.15, 1))
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_text_encoder_lora_weights(tracked_lora_weights, steps, title="Weights Norms of LoRAs over Steps for text encoder"):
    plt.figure(figsize=(12, 6))

    i = 0
    for lora_name, grad_norms in tracked_lora_weights.items():
        if i >= 282 :
            plt.plot(steps, grad_norms, label=lora_name)
        i+=1

    plt.xlabel("Training Step")
    plt.ylabel("Weights Norm")
    plt.title(title)
    plt.legend(loc="upper right", fontsize="small", bbox_to_anchor=(1.15, 1))
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_tuning_loss(tracked_tuning_loss, steps, title="tuning loss over steps"):
    plt.figure(figsize=(12, 6))

    plt.plot(steps,tracked_tuning_loss)


    plt.xlabel("Training Step")
    plt.ylabel("Tuning loss")
    plt.title(title)
    plt.legend(loc="upper right", fontsize="small", bbox_to_anchor=(1.15, 1))
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_ti_loss(ti_loss, steps, title="ti loss over steps"):
    plt.figure(figsize=(12, 6))

    plt.plot(steps,ti_loss)


    plt.xlabel("Training Step")
    plt.ylabel("Ti loss")
    plt.title(title)
    plt.legend(loc="upper right", fontsize="small", bbox_to_anchor=(1.15, 1))
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_ti_grad(ti_grad, steps, title="ti grad over steps"):
    plt.figure(figsize=(12, 6))

    plt.plot(steps,ti_grad)


    plt.xlabel("Training Step")
    plt.ylabel("Ti grad")
    plt.title(title)
    plt.legend(loc="upper right", fontsize="small", bbox_to_anchor=(1.15, 1))
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_ti_weights(ti_weights, steps, title="ti weights over steps"):
    plt.figure(figsize=(12, 6))


    for token, token_weights in ti_weights.items():
            plt.plot(steps,token_weights,label=token)

    plt.xlabel("Training Step")
    plt.ylabel("ti weights")
    plt.title(title)
    plt.legend(loc="upper right", fontsize="small", bbox_to_anchor=(1.15, 1))
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
"""
Execution

"""

In [ ]:
(
    placeholder_tokens,
    initializer_tokens,
    placeholder_token_ids,
    initializer_token_ids,
    text_encoder,
    unet,
    tokenizer,
    noise_scheduler,
    index_no_updates,
    inversion_dataloader,
    regularization_dataloader
) = prepare_dataset(
    instance_data_dir="./instance/",
    regularization_data_dir="./regularization/",
    pretrained_model_name_or_path="sd-legacy/stable-diffusion-v1-5",
    output_dir="./",
    revision=None,
    placeholder_tokens="<tok1>",
    initializer_tokens="character",
    seed=42,
    resolution=512,
    train_batch_size=2,
    device="cuda:0",
    token_map = {"ichigo" : "<tok1>"},
    safeloras_path = "./final_lora_1000.safetensors",
    do_pivotal_tuning  = True,
    do_regularization = True,
)


In [ ]:
tracked_tuning_loss, tracked_lora_grads,tracked_lora_weights,tracked_lora_paths = train(
    placeholder_tokens,
    initializer_tokens,
    placeholder_token_ids,
    initializer_token_ids,
    text_encoder,
    unet,
    tokenizer,
    noise_scheduler,
    index_no_updates,
    inversion_dataloader,
    regularization_dataloader = inversion_dataloader,
    max_train_steps_tuning=1000,
    max_train_steps_ti= 0,
    save_steps = 999,
    control_step = 10000,
    lora_rank= 4,
    lora_unet_target_modules={"UNet2DConditionModel"},
    lora_clip_target_modules={},
    learning_rate_unet  = 1e-4,
    learning_rate_text  = 0,
    learning_rate_ti = 0,
    lr_scheduler = "linear",
    lr_scheduler_lora = "linear",
    device="cuda:0",
    do_inversion = False,
    do_pivotal_tuning = True,
    do_regularization = True,
    out_name = "",
    output_dir = "./"
)

In [ ]:
"""
Plotting graphs
"""

In [ ]:
steps = [i for i in range(1008)]

In [ ]:
plot_unet_lora_grad_norms(tracked_lora_grads, steps, title="Gradient Norms of LoRAs over Steps for unet")

In [ ]:
plot_text_encoder_lora_grad_norms(tracked_lora_grads, steps, title="Gradient Norms of LoRAs over Steps for text_encoder")

In [ ]:
plot_unet_lora_weights(tracked_lora_weights, steps, title="Weights Norms of LoRAs over Steps for unet")

In [ ]:
plot_text_encoder_lora_weights(tracked_lora_weights, steps, title="Weights Norms of LoRAs over Steps for text encoder")

In [ ]:
plot_tuning_loss(tracked_tuning_loss, steps, title="tuning loss over steps")

In [ ]:
plot_ti_loss(ti_loss, steps, title="ti loss over steps")

In [ ]:
plot_ti_grad(ti_grad, steps, title="ti grad over steps")

In [ ]:
plot_ti_weights(ti_dist, steps, title="ti weights over steps")

In [ ]:
plot_ti_weights(ti_sim, steps, title="ti sim over steps")